In [ ]:
# =====================================================================
# Dynamic-Room LGBM Training — 17 features → outage
#   Uses linear_tree=True for smooth rx/ry extrapolation
#   Optuna 50 trials, 5-Fold CV, feasible-MAE objective
# =====================================================================

import numpy as np, lightgbm as lgb, optuna, json, time, os, glob
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error, r2_score
import warnings; warnings.filterwarnings('ignore')

# ============================================================
# 1. Load Data
# ============================================================
paths = ['dynamic_train_enriched.csv',
         '/kaggle/working/dynamic_train_enriched.csv'] + sorted(glob.glob('/kaggle/input/**/dynamic_train_enriched.csv', recursive=True))
fpath = None
for p in paths:
    if os.path.exists(p): fpath = p; break
if fpath is None: raise FileNotFoundError('dynamic_train_enriched.csv not found')

print(f'Loading {fpath}...')
data = np.loadtxt(fpath, delimiter=',', skiprows=1)
X = data[:, :-1]  # first 17 columns
Y = data[:, -1]   # outage
print(f'Loaded: {X.shape[0]} samples, {X.shape[1]} features')
print(f'Feasible (<10%): {(Y<0.10).sum()} ({(Y<0.10).sum()/len(Y)*100:.1f}%)')

# ============================================================
# 2. Train/Test Split
# ============================================================
np.random.seed(42)
idx = np.random.permutation(len(X))
sp = int(len(X)*0.85)
Xtr, Xte = X[idx[:sp]], X[idx[sp:]]
Ytr, Yte = Y[idx[:sp]], Y[idx[sp:]]

# ============================================================
# 3. Optuna Hyperparameter Tuning
# ============================================================
kf = KFold(n_splits=5, shuffle=True, random_state=42)

def objective(trial):
    params = {
        'objective': 'regression',
        'metric': 'mae',
        'verbosity': -1,
        'device': 'gpu',
        'linear_tree': True,
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'max_depth': trial.suggest_int('max_depth', 8, 16),
        'num_leaves': trial.suggest_int('num_leaves', 64, 256),
        'min_data_in_leaf': trial.suggest_int('min_data_in_leaf', 20, 80),
        'feature_fraction': trial.suggest_float('feature_fraction', 0.6, 0.95),
        'bagging_fraction': trial.suggest_float('bagging_fraction', 0.6, 0.9),
        'bagging_freq': 1,
    }
    maes = []
    for tr_idx, val_idx in kf.split(Xtr):
        Xt, Xv = Xtr[tr_idx], Xtr[val_idx]
        Yt, Yv = Ytr[tr_idx], Ytr[val_idx]
        model = lgb.LGBMRegressor(**params, n_estimators=500, early_stopping_round=30)
        model.fit(Xt, Yt, eval_set=[(Xv, Yv)])
        pred = model.predict(Xv)
        feas = Yv < 0.15
        if feas.sum() > 0:
            maes.append(mean_absolute_error(Yv[feas], pred[feas]))
        else:
            maes.append(mean_absolute_error(Yv, pred))
    return np.mean(maes)

print('\nOptuna BO-LGBM search (50 trials, 5-Fold CV, linear_tree=True)...')
study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=50, show_progress_bar=True)
print(f'\nBest params: {study.best_params}')
print(f'Best feasible MAE: {study.best_value*100:.2f}%')

# ============================================================
# 4. Train Final Model
# ============================================================
best = study.best_params
best['objective'] = 'regression'
best['metric'] = 'mae'
best['verbosity'] = -1
best['device'] = 'gpu'
best['linear_tree'] = True

print('\nTraining final model...')
model = lgb.LGBMRegressor(**best, n_estimators=1000, early_stopping_round=50)
model.fit(Xtr, Ytr, eval_set=[(Xte, Yte)])

pred = model.predict(Xte)
r2_all = r2_score(Yte, pred)
mae_all = mean_absolute_error(Yte, pred)
feas = Yte < 0.10
r2_feas = r2_score(Yte[feas], pred[feas]) if feas.sum() > 1 else 0
mae_feas = mean_absolute_error(Yte[feas], pred[feas]) if feas.sum() > 0 else 0

print(f'\nFinal LGBM metrics:')
print(f'  R² all:      {r2_all:.4f}')
print(f'  MAE all:     {mae_all*100:.2f}%')
print(f'  R² feasible: {r2_feas:.4f}')
print(f'  MAE feasible:{mae_feas*100:.2f}%')

# ============================================================
# 5. Save Model
# ============================================================
model.booster_.save_model('lgbm_dynamic.txt')
meta = {'n_features': X.shape[1], 'r2_all': r2_all, 'mae_feas': mae_feas,
        'best_params': best}
with open('lgbm_dynamic_meta.json', 'w') as f:
    json.dump(meta, f, indent=2)

print('\nModel saved: lgbm_dynamic.txt + lgbm_dynamic_meta.json')

from IPython.display import FileLink, display
display(FileLink('lgbm_dynamic.txt'))
display(FileLink('lgbm_dynamic_meta.json'))